##Setup

In [1]:
!pip install gymnasium shimmy ale-py
!pip install autorom
!AutoROM -y

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/bank_heist.bin
Inst

In [ ]:
from collections import defaultdict
import numpy as np
import cv2
import gymnasium as gym
from gymnasium.vector import SyncVectorEnv
import ale_py
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os

# ──────────────────────────────────────────────
# Rendering
# ──────────────────────────────────────────────
def render_env(envs, env_idx=0, iteration=0):
    frame = envs.envs[env_idx].render()
    save_dir = 'renders'
    os.makedirs(save_dir, exist_ok=True)
    plt.imshow(frame)
    plt.axis('off')
    plt.savefig(os.path.join(save_dir, f'frame_{iteration:05d}.png'))
    plt.close()

# ──────────────────────────────────────────────
# Go-Explore cell functions (pixel-based)
# ──────────────────────────────────────────────
def cellfn(frame):
    """Downsample frame to a coarse grayscale grid used as the cell key."""
    cell = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    cell = cv2.resize(cell, (11, 8), interpolation=cv2.INTER_AREA)
    cell = cell // 32
    return cell

def hashfn(cell):
    return hash(cell.tobytes())

# ──────────────────────────────────────────────
# Go-Explore archive
# ──────────────────────────────────────────────
e1 = 0.001
e2 = 0.00001

class Weights:
    times_chosen           = 0.1
    times_chosen_since_new = 0.0
    times_seen             = 0.3

class Powers:
    times_chosen           = 0.5
    times_chosen_since_new = 0.5
    times_seen             = 0.5

class Cell:
    def __init__(self):
        self.times_chosen           = 0
        self.times_chosen_since_new = 0
        self.times_seen             = 0

    def __setattr__(self, key, value):
        object.__setattr__(self, key, value)
        if key != 'score' and hasattr(self, 'times_seen'):
            self.score = self.cellscore()

    def cntscore(self, a):
        w = getattr(Weights, a)
        p = getattr(Powers, a)
        v = getattr(self, a)
        return w / (v + e1) ** p + e2

    def cellscore(self):
        return (self.cntscore('times_chosen') +
                self.cntscore('times_chosen_since_new') +
                self.cntscore('times_seen') + 1)

    def visit(self):
        self.times_seen += 1
        return self.times_seen == 1

    def choose(self):
        self.times_chosen           += 1
        self.times_chosen_since_new += 1
        return self.ram, self.reward, self.trajectory


# ──────────────────────────────────────────────
# Running return normaliser — keeps critic stable
# across wildly different return scales from
# different restored archive cells.
# ──────────────────────────────────────────────
class RunningNormaliser:
    def __init__(self, clip=10.0):
        self.mean  = 0.0
        self.var   = 1.0
        self.count = 0
        self.clip  = clip

    def update(self, x: torch.Tensor):
        batch_mean = x.mean().item()
        batch_var  = x.var().item() if x.numel() > 1 else 0.0
        batch_n    = x.numel()
        n          = self.count + batch_n
        delta      = batch_mean - self.mean
        self.mean  += delta * batch_n / n
        self.var    = (self.var * self.count + batch_var * batch_n +
                       delta ** 2 * self.count * batch_n / n) / n
        self.count  = n

    def normalise(self, x: torch.Tensor) -> torch.Tensor:
        self.update(x)
        return ((x - self.mean) / (self.var ** 0.5 + 1e-8)).clamp(-self.clip, self.clip)


# ──────────────────────────────────────────────
# Networks
# ──────────────────────────────────────────────
class Actor(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            self._feature_size = x.reshape(1, -1).size(1)
        self.fc = nn.Linear(self._feature_size, n_actions)

    def forward(self, x):
        x = x.float() / 255.
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return F.softmax(self.fc(x.reshape(x.size(0), -1)), dim=-1)


class Critic(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            self._feature_size = x.reshape(1, -1).size(1)
        self.fc = nn.Linear(self._feature_size, 1)

    def forward(self, x):
        x = x.float() / 255.
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(x.reshape(x.size(0), -1))


# ──────────────────────────────────────────────
# A3C Agent
# ──────────────────────────────────────────────
class A3CAgent:
    def __init__(self, envs, gamma=0.99, lr=1e-4, entropy_coef=0.05, grad_clip=0.5):
        self.num_envs          = envs.num_envs
        self.action_space_size = envs.single_action_space.n
        obs_shape              = envs.single_observation_space.shape  # (H, W, C)
        self.input_shape       = (obs_shape[2], obs_shape[0], obs_shape[1])  # (C, H, W)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {self.device}')

        self.actor  = Actor(self.input_shape, self.action_space_size).to(self.device)
        self.critic = Critic(self.input_shape).to(self.device)

        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr)

        self.gamma        = gamma
        self.entropy_coef = entropy_coef
        self.grad_clip    = grad_clip
        self.return_norm  = RunningNormaliser(clip=10.0)

    def preprocess(self, states):
        """(N, H, W, C) numpy → (N, C, H, W) float tensor on device."""
        return torch.from_numpy(states).permute(0, 3, 1, 2).to(self.device)

    def select_action(self, states):
        with torch.no_grad():
            probs = self.actor(self.preprocess(states))
        actions = torch.distributions.Categorical(probs).sample()
        return actions.cpu().numpy()

    def update_model(self, states_batch, actions_batch, rewards_batch,
                     dones_batch, policy_mask_batch):
        """
        policy_mask_batch : list of (num_envs,) bool arrays
            True  → step used the policy   → include in actor loss
            False → step was random        → critic only, skip actor loss
        """
        num_steps = len(states_batch)

        flat_states  = torch.cat([self.preprocess(s) for s in states_batch], dim=0)
        flat_actions = torch.tensor(np.concatenate(actions_batch),
                                    dtype=torch.long).to(self.device)
        flat_rewards = torch.tensor(np.concatenate(rewards_batch),
                                    dtype=torch.float32).to(self.device)
        flat_dones   = torch.tensor(np.concatenate(dones_batch).astype(np.float32),
                                    ).to(self.device)
        flat_mask    = torch.tensor(np.concatenate(policy_mask_batch),
                                    dtype=torch.bool).to(self.device)

        # ── Discounted returns per environment ──────────────────────────────
        reshaped_rewards = flat_rewards.view(num_steps, self.num_envs)
        reshaped_dones   = flat_dones.view(num_steps,   self.num_envs)

        all_returns = []
        for env_idx in range(self.num_envs):
            R = 0.0
            env_returns = []
            for t in reversed(range(num_steps)):
                if reshaped_dones[t, env_idx]:
                    R = 0.0
                R = reshaped_rewards[t, env_idx].item() + self.gamma * R
                env_returns.insert(0, R)
            all_returns.extend(env_returns)

        returns_raw  = torch.tensor(all_returns, dtype=torch.float32).to(self.device)
        # Global normalisation so critic sees a stable target regardless of
        # which archive cell was restored (returns vary from 0 to 400+).
        returns_norm = self.return_norm.normalise(returns_raw)

        # ── Critic — Huber loss is robust to occasional outlier returns ──────
        values      = self.critic(flat_states).squeeze(-1)
        critic_loss = F.huber_loss(values, returns_norm, delta=1.0)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), self.grad_clip)
        self.critic_optimizer.step()

        # ── Actor — policy steps only; don't reinforce random actions ────────
        if flat_mask.sum() == 0:
            return 0.0, critic_loss.item(), 0.0

        policy_states  = flat_states[flat_mask]
        policy_actions = flat_actions[flat_mask]
        policy_returns = returns_norm[flat_mask]

        current_probs = self.actor(policy_states)
        dist          = torch.distributions.Categorical(current_probs)
        log_probs     = dist.log_prob(policy_actions)
        entropy       = dist.entropy().mean()

        policy_values = self.critic(policy_states).squeeze(-1).detach()
        advantages    = policy_returns - policy_values
        advantages    = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        actor_loss = -(log_probs * advantages).mean() - self.entropy_coef * entropy

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), self.grad_clip)
        self.actor_optimizer.step()

        return actor_loss.item(), critic_loss.item(), entropy.item()


# ──────────────────────────────────────────────
# State restoration
# ──────────────────────────────────────────────
def restore_single_env(env, ram):
    """
    Restore ALE state and return the current screen observation.
    Uses getScreenRGB() directly so we never call env.reset(),
    which would wipe the restored state.
    """
    env.unwrapped.ale.restoreState(ram)
    return env.unwrapped.ale.getScreenRGB()   # (210, 160, 3) uint8


def select_cells_from_archive(archive, num_cells):
    """Score-weighted sampling — each env may get a different cell."""
    hashes = list(archive.keys())
    scores = np.array([archive[h].score for h in hashes])
    probs  = scores / scores.sum()
    chosen = np.random.choice(len(hashes), size=num_cells, replace=True, p=probs)
    return [hashes[i] for i in chosen]


# ──────────────────────────────────────────────
# Hyperparameters
# ──────────────────────────────────────────────
NUM_ENVS         = 4
STEPS_PER_UPDATE = 128
WARMUP_ITERS     = 100   # Pure random exploration before A3C kicks in

EPSILON_START = 1.0
EPSILON_END   = 0.05
EPSILON_DECAY = 0.995   # Per iteration after warmup

archive   = defaultdict(lambda: Cell())
highscore = 0
frames    = 0
epsilon   = EPSILON_START

def make_env():
    return gym.make('ALE/MontezumaRevenge-v5', render_mode='rgb_array')

envs = SyncVectorEnv([make_env for _ in range(NUM_ENVS)])

current_scores = [0.0] * NUM_ENVS
trajectories   = [[]   for _ in range(NUM_ENVS)]
current_lives  = [6    for _ in range(NUM_ENVS)]

agent      = A3CAgent(envs)
iterations = 0

current_states, _ = envs.reset()

# ──────────────────────────────────────────────
# Main loop
# ──────────────────────────────────────────────
while True:

    # ── Restore each env to a (possibly different) archived cell ─────────────
    if iterations > 0 and len(archive) > 0:
        chosen_hashes = select_cells_from_archive(archive, NUM_ENVS)
        obs_list = []
        for i, h in enumerate(chosen_hashes):
            cell             = archive[h]
            ram, score, traj = cell.choose()
            obs              = restore_single_env(envs.envs[i], ram)
            obs_list.append(obs)
            current_scores[i] = score
            trajectories[i]   = traj.copy()
            current_lives[i]  = 6
        current_states = np.stack(obs_list)   # (NUM_ENVS, H, W, C)
    else:
        current_states, _ = envs.reset()
        current_scores    = [0.0] * NUM_ENVS
        trajectories      = [[]   for _ in range(NUM_ENVS)]
        current_lives     = [6    for _ in range(NUM_ENVS)]

    # ── Collect rollout ──────────────────────────────────────────────────────
    episode_states       = []
    episode_actions      = []
    episode_rewards      = []
    episode_dones        = []
    episode_policy_masks = []

    # Decide once per rollout so the entire trajectory is consistent
    # (fully random or fully on-policy — no mixed gradients within a batch).
    use_random = (iterations < WARMUP_ITERS) or (random.random() < epsilon)

    for step in range(STEPS_PER_UPDATE):
        if use_random:
            actions     = np.array([envs.single_action_space.sample()
                                    for _ in range(NUM_ENVS)])
            policy_mask = np.zeros(NUM_ENVS, dtype=bool)
        else:
            actions     = agent.select_action(current_states)
            policy_mask = np.ones(NUM_ENVS, dtype=bool)

        next_states, rewards, terminals, truncations, infos = envs.step(actions)

        # Per-env life tracking — infos['lives'] after SyncVectorEnv auto-reset
        # shows the post-reset value (6), not the terminal value.
        # We maintain current_lives ourselves to detect a genuine life loss.
        lives_info = infos.get('lives', None)
        for env_idx in range(NUM_ENVS):
            env_lives = (int(lives_info[env_idx])
                         if lives_info is not None
                         else current_lives[env_idx])
            if env_lives < current_lives[env_idx]:
                terminals[env_idx] = True
            current_lives[env_idx] = (6 if (terminals[env_idx] or truncations[env_idx])
                                      else env_lives)

        dones = terminals | truncations

        episode_states.append(current_states.copy())
        episode_actions.append(actions)
        episode_rewards.append(rewards)
        episode_dones.append(dones)
        episode_policy_masks.append(policy_mask)

        # ── Archive update ───────────────────────────────────────────────────
        for env_idx in range(NUM_ENVS):
            current_scores[env_idx] += rewards[env_idx]
            trajectories[env_idx].append(int(actions[env_idx]))
            frames += 1

            if current_scores[env_idx] > highscore:
                highscore = current_scores[env_idx]

            # Only update archive when the env is still alive
            if not (terminals[env_idx] or truncations[env_idx]):
                cell_obs    = cellfn(next_states[env_idx])
                cellhash    = hashfn(cell_obs)
                cell        = archive[cellhash]
                first_visit = cell.visit()

                cell_reward = getattr(cell, 'reward',     -1e9)
                cell_traj   = getattr(cell, 'trajectory', [])
                better  = current_scores[env_idx] > cell_reward
                shorter = (current_scores[env_idx] == cell_reward and
                           len(trajectories[env_idx]) < len(cell_traj))

                if first_visit or better or shorter:
                    cell.ram        = envs.envs[env_idx].unwrapped.ale.cloneState()
                    cell.reward     = current_scores[env_idx]
                    cell.trajectory = trajectories[env_idx].copy()
                    cell.times_chosen           = 0
                    cell.times_chosen_since_new = 0

            if dones[env_idx]:
                current_scores[env_idx] = 0.0
                trajectories[env_idx]   = []

        current_states = next_states

    # ── A3C update ────────────────────────────────────────────────────────────
    if iterations >= WARMUP_ITERS:
        actor_loss, critic_loss, entropy = agent.update_model(
            episode_states, episode_actions, episode_rewards,
            episode_dones, episode_policy_masks
        )
        epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    else:
        actor_loss = critic_loss = entropy = 0.0

    iterations += 1

    if iterations % 10 == 0:
        print(
            f"Iter: {iterations:5d} | Cells: {len(archive):5d} | "
            f"Frames: {frames:8d} | MaxReward: {highscore:.1f} | "
            f"Eps: {epsilon:.3f} | "
            f"ALoss: {actor_loss:8.4f} | CLoss: {critic_loss:8.4f} | "
            f"Entropy: {entropy:.4f}"
        )

    if iterations % 50 == 0:
        render_env(envs, env_idx=0, iteration=iterations)

Using device: cuda
Iter:    10 | Cells:   118 | Frames:     5120 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    20 | Cells:   151 | Frames:    10240 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    30 | Cells:   175 | Frames:    15360 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    40 | Cells:   183 | Frames:    20480 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    50 | Cells:   188 | Frames:    25600 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    60 | Cells:   193 | Frames:    30720 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    70 | Cells:   193 | Frames:    35840 | MaxReward: 0.0 | Eps: 1.000 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:    80 | Cells:   196 | Frames:    40960 | MaxReward: 0.0 | Eps: 1